The LightCGN model comes from the paper “LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation” (Xiangnan He, Kuan Deng, Xiang Wang, Yan Li, YongDong Zhang, and Meng Wang. 2020). To implement the lightCGN model, parts of the code were taken from the official implementation of the paper [1] and the RecBole project implementation [2]. We used an RMSE loss function given in the notebook of the project, and implemented a custom wrapper designed to integrate the LightGCN model with the scikit-learn framework, enabling hyperparameter tuning, model training, and evaluation. Each try is saved in a log file, along with the parameters and the loss (on the training set and testing set).

We implemented several strategies to prevent overfitting in the lightCGNWrapper class: L2 Regularization and early stopping (a patience parameter allows the model to stop training early if the validation loss does not improve for a certain number of epochs).

You can hypertune the following parameters by adding values in the parameter grid:
-	Embedding Size (emb_size): affects the capacity of the latent representations of users and items.
-	Number of Layers (n_layers): determines how many times the graph convolution operation is applied, influencing the depth of information propagation.
-	Learning Rate (lr)
-	Number of Epochs (epochs)
-	L2 Regularization (l2_reg)


[1] https://github.com/gusye1234/LightGCN-PyTorch
[2] https://github.com/RUCAIBox/RecBole/blob/master/recbole/model/general_recommender/lightgcn.py


In [ ]:
# @title Set ONLINE=False if running locally
ONLINE = True

In [ ]:
"""First, we import the dataset and everything needed from the notebook given in class"""

import math
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import mean_squared_error
import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn
import torch.nn.functional as F
import gdown
import scipy.sparse as sp
import os

# Define device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using {} device".format(device))

# Load data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)

data_pd = pd.read_csv('data_train.csv')
train_pd, test_pd = train_test_split(data_pd, train_size=0.9, random_state=42)

def extract_users_items_predictions(data_pd):
    users, movies = \
        [np.squeeze(arr) for arr in np.split(data_pd.Id.str.extract('r(\d+)_c(\d+)').values.astype(int) - 1, 2, axis=-1)]
    predictions = data_pd.Prediction.values
    return users, movies, predictions

train_users, train_movies, train_predictions = extract_users_items_predictions(train_pd)
test_users, test_movies, test_predictions = extract_users_items_predictions(test_pd)

In [ ]:
"""same RMSE loss function as the one in the notebook"""

class RMSELoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.mse = nn.MSELoss()
        self.eps = eps

    def forward(self, yhat, y):
        loss = torch.sqrt(self.mse(yhat, y) + self.eps)
        return loss

"""The RatingsDataset is used to create dataset objects for the training and test sets.
Then they are given to a DataLoader to handle batching, shuffling, and parallel data loading"""

class RatingsDataset(Dataset):
    def __init__(self, users, items, ratings):
        self.users = torch.tensor(users, dtype=torch.long)
        self.items = torch.tensor(items, dtype=torch.long)
        self.ratings = torch.tensor(ratings, dtype=torch.float)

    def __len__(self):
      #Returns the total number of samples in the dataset.
        return len(self.ratings)

    def __getitem__(self, idx):
        """Retrieves the sample at index `idx`.

        Args:
            idx (int): Index of the sample to retrieve.

        Returns:
            dict: A dictionary containing user index, item index, and rating."""

        return self.users[idx], self.items[idx], self.ratings[idx]


In [ ]:
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_size, n_layers):
        super(LightGCN, self).__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.emb_size = emb_size
        self.n_layers = n_layers

        self.embedding_user = nn.Embedding(num_embeddings=self.n_users, embedding_dim=self.emb_size)
        self.embedding_item = nn.Embedding(num_embeddings=self.n_items, embedding_dim=self.emb_size)

        nn.init.normal_(self.embedding_user.weight, std=0.1)
        nn.init.normal_(self.embedding_item.weight, std=0.1)

        self.A_norm = self.get_norm_adj_mat()


    def get_norm_adj_mat(self):

        """
        Get the normalized interaction matrix of users and items.

        Construct the square matrix from the training data and normalize it

        """

        A = sp.dok_matrix((self.n_users + self.n_items, self.n_users + self.n_items), dtype=np.float32)
        A = A.tolil()
        R = sp.coo_matrix((train_predictions, (train_users, train_movies)), shape=(self.n_users, self.n_items))
        A[:self.n_users, self.n_users:] = R
        A[self.n_users:, :self.n_users] = R.T

        rowsum = np.array(A.sum(axis=1))
        D_inv = np.power(rowsum, -0.5).flatten()
        D_inv[np.isinf(D_inv)] = 0.0
        D = sp.diags(D_inv)

        A_norm = D.dot(A).dot(D)
        A_norm = A_norm.tocsr()
        A_norm = self._convert_sp_mat_to_sp_tensor(A_norm)

        return A_norm

    def _convert_sp_mat_to_sp_tensor(self, X):
        coo = X.tocoo()
        indices = np.vstack((coo.row, coo.col)).astype(np.int64)
        values = coo.data.astype(np.float32)
        shape = coo.shape

        indices = torch.tensor(indices, dtype=torch.long, device=device)
        values = torch.tensor(values, dtype=torch.float32, device=device)
        size = torch.Size(shape)

        return torch.sparse.FloatTensor(indices, values, size).to(device)

    def get_ego_embeddings(self):

        """Get the embedding of users and items and combine to an embedding matrix.

        Returns:
            Tensor of the embedding matrix. Shape of [n_items+n_users, embedding_dim]
        """
        user_embeddings = self.embedding_user.weight
        item_embeddings = self.embedding_item.weight
        ego_embeddings = torch.cat([user_embeddings, item_embeddings], dim=0)
        return ego_embeddings

    def forward(self, users_items):

        #Initial Embeddings
        all_embeddings = self.get_ego_embeddings()
        embeddings_list = [all_embeddings]

        #Graph Convolution Layers
        for layer_idx in range(self.n_layers):
            all_embeddings = torch.sparse.mm(self.A_norm, all_embeddings)
            embeddings_list.append(all_embeddings)

        #Aggregation
        lightgcn_all_embeddings = torch.stack(embeddings_list, dim=1)
        lightgcn_all_embeddings = torch.mean(lightgcn_all_embeddings, dim=1)

        #Splitting User and Item Embeddings
        user_all_embeddings, item_all_embeddings = torch.split(
            lightgcn_all_embeddings, [self.n_users, self.n_items]
        )


        #Extracting Batch Embeddings
        users, items = users_items[:, 0], users_items[:, 1]
        user_embeddings = user_all_embeddings[users]
        item_embeddings = item_all_embeddings[items]

        #The final recommendation score for a user-item pair is computed as the inner product of their embedding
        scores = (user_embeddings * item_embeddings).sum(dim=1)
        return scores

In [ ]:
""" The LightGCNWrapper class is a custom wrapper designed to integrate the LightGCN model with scikit-learn's grid search functionality. """

class LightGCNWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, emb_size=256, n_layers=4, lr=0.001, epochs=43, l2_reg=1e-6, log_file='results_log.csv', test_data=None, test_targets=None, patience=5):

      """
      Initializes the LightGCNWrapper with given hyperparameters.

      Args:
          emb_size (int): The size of the embedding vectors for users and items.
          n_layers (int): The number of graph convolutional layers.
          lr (float): The learning rate for the optimizer.
          epochs (int): The number of training epochs.
          l2_reg (float): The L2 regularization parameter.
          log_file (str): The path to the log file for saving training and test results.
          test_data (np.array): The test data for evaluation.
          test_targets (np.array): The test targets for evaluation.
          patience (int): The number of epochs to wait for improvement before early stopping.
      """

      self.emb_size = emb_size
      self.n_layers = n_layers
      self.lr = lr
      self.epochs = epochs
      self.l2_reg = l2_reg
      self.log_file = log_file
      self.test_data = test_data
      self.test_targets = test_targets
      self.patience = patience

    def fit(self, X, y):

        """
        Trains the LightGCN model on the given training data.

        Args:
            X (np.array): The training data containing user-item pairs.
            y (np.array): The training targets (ratings).

        Returns:
            self: The trained LightGCNWrapper instance.
        """
        n_users = np.max(X[:, 0]) + 1
        n_items = np.max(X[:, 1]) + 1
        self.model = LightGCN(n_users, n_items, self.emb_size, self.n_layers).to(device)
        self.criterion = RMSELoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.l2_reg)

        self.model.train()
        dataset = RatingsDataset(X[:, 0], X[:, 1], y)
        loader = DataLoader(dataset, batch_size=1024, shuffle=True)

        best_loss = float('inf')
        epochs_no_improve = 0

        for epoch in range(self.epochs):
            total_loss = 0
            self.model.train()
            for batch in loader:
                batch_users = batch[0].to(device)
                batch_items = batch[1].to(device)
                batch_ratings = batch[2].float().to(device)
                batch_data = torch.stack((batch_users, batch_items), dim=1)

                self.optimizer.zero_grad()
                scores = self.model(batch_data)
                loss = self.criterion(scores, batch_ratings)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()

            train_loss = total_loss / len(loader)

            #skip evaluation if not testing set
            if self.test_data is not None and self.test_targets is not None:
                test_loss = self.evaluate(self.test_data, self.test_targets)
            else:
                test_loss = None

            # Log the results
            with open(self.log_file, 'a') as f:
                f.write(f"{self.emb_size},{self.n_layers},{self.lr},{self.epochs},{self.l2_reg},{train_loss},{test_loss}\n")

            if test_loss is not None and test_loss < best_loss:
                best_loss = test_loss
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

            #skip early stopping if no testing set
            if test_loss is not None and epochs_no_improve >= self.patience:
                print(f"Early stopping after {epoch+1} epochs")
                break

        return self

    def predict(self, X):

      """
      Predicts ratings for the given user-item pairs.

      Args:
          X (np.array): The data containing user-item pairs.

      Returns:
          np.array: The predicted ratings.
      """
      self.model.eval()
      with torch.no_grad():
          users = torch.tensor(X[:, 0], dtype=torch.long).to(device)
          items = torch.tensor(X[:, 1], dtype=torch.long).to(device)
          batch_data = torch.stack((users, items), dim=1)
          scores = self.model(batch_data)
      return scores.cpu().numpy()

    def evaluate(self, X, y):
      """
        Evaluates the model on the given test data.

        Args:
            X (np.array): The test data containing user-item pairs.
            y (np.array): The test targets (ratings).

        Returns:
            float: The RMSE loss on the test data.
       """
      self.model.eval()
      dataset = RatingsDataset(X[:, 0], X[:, 1], y)
      loader = DataLoader(dataset, batch_size=1024, shuffle=False)
      total_loss = 0
      with torch.no_grad():
          for batch in loader:
              batch_users = batch[0].to(device)
              batch_items = batch[1].to(device)
              batch_ratings = batch[2].float().to(device)
              batch_data = torch.stack((batch_users, batch_items), dim=1)

              scores = self.model(batch_data)
              loss = self.criterion(scores, batch_ratings)
              total_loss += loss.item()
      return total_loss / len(loader)


This cell performs the grid search : skip it if you already know the best parameters (it takes very long)

In [ ]:
with open('results_log.csv', 'w') as f:
    f.write("emb_size,n_layers,lr,epochs,l2_reg,train_loss,test_loss\n")

# Define the parameter grid
param_grid = {
    'emb_size': [64, 128, 256],
    'n_layers': [3,4,5],
    'lr': [0.01,0.001],
    'epochs': [30,40,50],
    'l2_reg': [1e-5,1e-6]
}

# Prepare the data
train_data = np.column_stack((train_users, train_movies))
train_targets = train_predictions
test_data = np.column_stack((test_users, test_movies))
test_targets = test_predictions

# Create the grid search object
grid_search = GridSearchCV(LightGCNWrapper(log_file='results_log.csv', test_data=test_data, test_targets=test_targets), param_grid, cv=3, scoring='neg_root_mean_squared_error', verbose=2)

# Perform the grid search
grid_search.fit(train_data, train_targets)

# Save the results
results = pd.read_csv('results_log.csv')
results.to_csv('grid_search_results.csv', index=False)
print("Best parameters found: ", grid_search.best_params_)
print("Best RMSE: ", -grid_search.best_score_)

Test on the submission file, output a file with the predictions ; we retrain the model with the best parameters from the grid search (model training is around 51min with an L4 GPU)




In [ ]:
# Assuming you have the best parameters from the grid search
best_params = {
    'emb_size': 256,
    'n_layers': 4,
    'lr': 0.001,
    'epochs': 43,
    'l2_reg': 1e-6
}

#test_data = np.column_stack((test_users, test_movies))
#test_targets = test_predictions

# Step 1: Initialize and retrain the best model with the full training data
best_model = LightGCNWrapper(
    emb_size=best_params['emb_size'],
    n_layers=best_params['n_layers'],
    lr=best_params['lr'],
    epochs=best_params['epochs'],
    l2_reg=best_params['l2_reg'],
    test_data=None,  # No test data needed for training
    test_targets=None,  # No test targets needed for training
    log_file='retrain_results_log.csv'
)


# Load the full training data
full_train_users_items = np.stack((train_users, train_movies), axis=1)
full_train_targets = train_predictions


# Fit the model on the full training data
best_model.fit(full_train_users_items, full_train_targets)

if ONLINE:
    url2 = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url2, output, fuzzy=True)


# Load the sample submission file
sample_submission = pd.read_csv('sampleSubmission.csv')
# Extract user and item IDs from the 'Id' column
sample_submission['user_id'] = sample_submission['Id'].apply(lambda x: int(x.split('_')[0][1:]) - 1)
sample_submission['item_id'] = sample_submission['Id'].apply(lambda x: int(x.split('_')[1][1:]) - 1)

# Prepare the data for prediction
user_ids = sample_submission['user_id'].values
item_ids = sample_submission['item_id'].values
user_item_pairs = np.stack((user_ids, item_ids), axis=1)

# Convert user-item pairs to PyTorch tensors
user_item_pairs_tensor = torch.tensor(user_item_pairs, dtype=torch.long).to(device)

# Predict the ratings using the retrained model
predictions = best_model.predict(user_item_pairs_tensor.cpu().numpy())

# Ensure predictions are within the valid range (e.g., if ratings should be between 1 and 5)
challenge_predictions = np.clip(predictions, 1, 5)

# Save the predictions to a new CSV file
sample_submission['Prediction'] = challenge_predictions

csv_path = 'csvs'
if not os.path.exists(csv_path):
    os.mkdir(csv_path)

sample_submission[['Id', 'Prediction']].to_csv(os.path.join(csv_path, 'submission_gcn.csv'), index=False)

print("Predictions have been saved")

Test now on the testing dataset : the model is already trained

In [ ]:
def calculate_rmse(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    return rmse

test_data = np.stack((test_users, test_movies), axis=1)
test_targets = np.array(test_predictions)

# Convert test data to PyTorch tensors
test_user_item_pairs_tensor = torch.tensor(test_data, dtype=torch.long).to(device)

# Predict the ratings using the trained model
predictions = best_model.predict(test_data).clip(1, 5)

# Calculate RMSE loss
rmse_loss = calculate_rmse(test_targets, predictions)

print(f"RMSE Loss on the test set: {rmse_loss}")

os.path.join(csv_path, 'submission_als.csv')

predictions_path = 'predictions'
if not os.path.exists(predictions_path):
    os.mkdir(predictions_path)

np.savez_compressed(os.path.join(predictions_path, 'gcn_predictions'), test=predictions,
                        challenge=challenge_predictions)